# The Impact of Generative AI on Student Academic Outcomes

## Research Questions

1. **RQ1 — Substitution & GPA:** Does GenAI use displace traditional study time, and does that substitution affect Post-semester GPA?
2. **RQ2 — Anxiety & Burnout:** What factors best predict exam anxiety and burnout risk — AI dependency, usage intensity, or institutional policy?
3. **RQ3 — Smart AI Use & Retention:** Does "smart" AI use (prompt engineering skill + tool diversity) lead to better skill retention, and does institutional policy moderate this effect?

## Data Source

**File:** `ai_student_impact_dataset.csv`  
**Size:** 50,000 students × 16 variables  
**Variables:** Demographics (major, year), AI usage (hours, use case, skill, tools), academic outcomes (GPA pre/post, skill retention), mental health (dependency, anxiety, burnout), institutional context (policy).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# ── Load data ──────────────────────────────────────────────────
DATA_PATH = '../ai_student_impact_dataset (1).csv'
df = pd.read_csv(DATA_PATH)

# Derived columns used throughout
df['GPA_Change'] = df['Post_Semester_GPA'] - df['Pre_Semester_GPA']
df['GenAI_Bucket'] = pd.cut(
    df['Weekly_GenAI_Hours'],
    bins=[-1, 3, 12, 40],
    labels=['Low\n(<3h)', 'Mid\n(3-12h)', 'High\n(>12h)']
)

print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

## RQ1: Does GenAI Replace Traditional Study Time, and Does That Hurt GPA?

We first examine whether heavier GenAI use is associated with reduced traditional study hours (a substitution hypothesis), and whether that substitution translates into lower Post-semester GPA.

In [ ]:
# ── RQ1: Correlations ──────────────────────────────────────────
r_trad_gpa, p_trad = stats.pearsonr(df['Traditional_Study_Hours'], df['Post_Semester_GPA'])
r_ai_gpa,   p_ai   = stats.pearsonr(df['Weekly_GenAI_Hours'],      df['Post_Semester_GPA'])
r_ai_trad,  p_at   = stats.pearsonr(df['Weekly_GenAI_Hours'],      df['Traditional_Study_Hours'])

print("Pearson Correlations:")
print(f"  Traditional hours  ↔ Post GPA  : r = {r_trad_gpa:+.4f}  (p = {p_trad:.2e})")
print(f"  GenAI hours        ↔ Post GPA  : r = {r_ai_gpa:+.4f}  (p = {p_ai:.2e})")
print(f"  GenAI hours        ↔ Trad hours: r = {r_ai_trad:+.4f}  (p = {p_at:.2e})")

# ANOVA across GenAI buckets
bucket_groups = [g['Post_Semester_GPA'].values
                 for _, g in df.groupby('GenAI_Bucket', observed=True)]
F, p_anova = stats.f_oneway(*bucket_groups)
print(f"\nOne-way ANOVA (Post GPA by GenAI Bucket): F = {F:.2f},  p = {p_anova:.2e}")

print("\nGroup summary:")
print(df.groupby('GenAI_Bucket', observed=True).agg(
    n=('Student_ID','count'),
    avg_trad_hours=('Traditional_Study_Hours','mean'),
    avg_post_gpa=('Post_Semester_GPA','mean'),
    avg_gpa_change=('GPA_Change','mean')
).round(3).to_string())

In [ ]:
# ── RQ1: Figure ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
COLORS = ['#2196F3', '#FF9800', '#F44336']

bucket_trad = df.groupby('GenAI_Bucket', observed=True)['Traditional_Study_Hours'].mean()
bucket_gpa  = df.groupby('GenAI_Bucket', observed=True)['Post_Semester_GPA'].mean()

bucket_trad.plot(kind='bar', ax=axes[0], color=COLORS, edgecolor='black', width=0.6)
axes[0].set_title('Avg Traditional Study Hours\nby GenAI Usage Level', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Weekly GenAI Hours', fontsize=11)
axes[0].set_ylabel('Avg Traditional Study Hours / Week', fontsize=11)
axes[0].tick_params(axis='x', rotation=0)

bucket_gpa.plot(kind='bar', ax=axes[1], color=COLORS, edgecolor='black', width=0.6)
axes[1].set_title('Post-Semester GPA\nby GenAI Usage Level', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Weekly GenAI Hours', fontsize=11)
axes[1].set_ylabel('Mean Post-Semester GPA', fontsize=11)
axes[1].set_ylim(3.28, 3.40)
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('RQ1: GenAI Usage, Study Displacement, and GPA', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('rq1_genai_study_gpa.pdf', bbox_inches='tight')
plt.savefig('rq1_genai_study_gpa.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as rq1_genai_study_gpa.pdf")

**RQ1 Interpretation:** A significant but mild displacement effect exists (r = −0.16). Moderate GenAI users (3–12h/week) have the *highest* Post-GPA (3.37) — suggesting an inverted-U relationship where some AI use complements study while heavy use (>12h) begins to erode traditional study time enough to harm outcomes. Prior GPA remains the dominant predictor (r = 0.93).

---

## RQ2: What Predicts Exam Anxiety and Burnout?

We examine AI dependency, usage intensity, and institutional policy as predictors of exam anxiety and burnout risk.

In [ ]:
# ── RQ2: Correlations & Group Tests ────────────────────────────
r_dep_anx, p_da = stats.pearsonr(df['Perceived_AI_Dependency'], df['Anxiety_Level_During_Exams'])
r_ai_anx,  p_aa = stats.pearsonr(df['Weekly_GenAI_Hours'],      df['Anxiety_Level_During_Exams'])

print("Correlations with Exam Anxiety:")
print(f"  Perceived AI Dependency ↔ Anxiety: r = {r_dep_anx:+.4f}  (p = {p_da:.2e})")
print(f"  Weekly GenAI Hours      ↔ Anxiety: r = {r_ai_anx:+.4f}  (p = {p_aa:.2e})")

policy_groups = [g['Anxiety_Level_During_Exams'].values
                 for _, g in df.groupby('Institutional_Policy')]
H, p_kw = stats.kruskal(*policy_groups)
print(f"\nKruskal-Wallis (Anxiety across policies): H = {H:.2f},  p = {p_kw:.2e}")

print("\nAnxiety by Institutional Policy:")
print(df.groupby('Institutional_Policy')['Anxiety_Level_During_Exams']
      .agg(['mean', 'std', 'count']).round(3).to_string())

print("\nBurnout Risk (%) by Institutional Policy:")
bp = df.groupby(['Institutional_Policy','Burnout_Risk_Level']).size().unstack(fill_value=0)
print((bp.div(bp.sum(axis=1), axis=0) * 100).round(1).to_string())

In [ ]:
# ── RQ2: Figure ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: mean anxiety by policy
anx_means = df.groupby('Institutional_Policy')['Anxiety_Level_During_Exams'].mean()
anx_errs  = df.groupby('Institutional_Policy')['Anxiety_Level_During_Exams'].sem()
policy_colors = ['#4CAF50', '#2196F3', '#F44336']
anx_means.plot(kind='bar', ax=axes[0], color=policy_colors, edgecolor='black',
               width=0.6, yerr=anx_errs, capsize=4)
axes[0].set_title('Mean Exam Anxiety\nby Institutional Policy', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Institutional Policy', fontsize=11)
axes[0].set_ylabel('Mean Anxiety Score (1–10)', fontsize=11)
axes[0].set_ylim(3.8, 5.4)
axes[0].tick_params(axis='x', rotation=15)

# Right: burnout stacked bar by policy
bp = df.groupby(['Institutional_Policy','Burnout_Risk_Level']).size().unstack(fill_value=0)
bp_pct = bp.div(bp.sum(axis=1), axis=0) * 100
bp_pct[['High','Medium','Low']].plot(
    kind='bar', stacked=True, ax=axes[1],
    color=['#F44336','#FF9800','#4CAF50'], edgecolor='black')
axes[1].set_title('Burnout Risk Distribution\nby Institutional Policy', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Institutional Policy', fontsize=11)
axes[1].set_ylabel('Percentage of Students (%)', fontsize=11)
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(title='Burnout Risk', bbox_to_anchor=(1.02,1), loc='upper left')

plt.suptitle('RQ2: Anxiety and Burnout by Institutional Policy', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('rq2_anxiety_burnout.pdf', bbox_inches='tight')
plt.savefig('rq2_anxiety_burnout.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as rq2_anxiety_burnout.pdf")

**RQ2 Interpretation:** AI dependency is the strongest predictor of anxiety (r = +0.31). The policy finding is counterintuitive: **strict bans produce higher anxiety (4.89 vs 4.12) and more high-burnout students (29.8% vs 23.8%)**. This suggests that prohibition — rather than reducing the stress associated with AI — may drive covert usage under fear of academic sanction, compounding anxiety.

---

## RQ3: Does "Smart" AI Use Improve Skill Retention?

We test whether prompt engineering skill and tool diversity (quality dimensions of AI use) predict skill retention scores, and whether institutional policy moderates these effects.

In [ ]:
# ── RQ3: Correlations & ANOVA ──────────────────────────────────
r_div_ret, p_dr = stats.pearsonr(df['Tool_Diversity'], df['Skill_Retention_Score'])
print(f"Tool Diversity ↔ Skill Retention: r = {r_div_ret:+.4f}  (p = {p_dr:.2e})")

skill_groups = [g['Skill_Retention_Score'].values
                for _, g in df.groupby('Prompt_Engineering_Skill')]
F2, p2 = stats.f_oneway(*skill_groups)
print(f"\nANOVA (Retention by Prompt Skill): F = {F2:.1f},  p = {p2:.2e}")

print("\nRetention by Prompt Engineering Skill:")
print(df.groupby('Prompt_Engineering_Skill')['Skill_Retention_Score']
      .agg(['mean','std']).round(3).to_string())

print("\nRetention by Tool Diversity (1–5 tools):")
print(df.groupby('Tool_Diversity')['Skill_Retention_Score']
      .agg(['mean','std','count']).round(3).to_string())

print("\nRetention: Prompt Skill × Policy (interaction table):")
print(df.pivot_table(
    values='Skill_Retention_Score',
    index='Prompt_Engineering_Skill',
    columns='Institutional_Policy',
    aggfunc='mean').round(2).to_string())

In [ ]:
# ── RQ3: Figure ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Retention by Prompt Skill × Policy (grouped bar)
pivot = df.pivot_table(values='Skill_Retention_Score',
                       index='Prompt_Engineering_Skill',
                       columns='Institutional_Policy', aggfunc='mean')
skill_order = ['Beginner', 'Intermediate', 'Advanced']
pivot.loc[skill_order].plot(kind='bar', ax=axes[0], edgecolor='black', width=0.7)
axes[0].set_title('Skill Retention by Prompt Engineering Skill\n× Institutional Policy',
                  fontweight='bold', fontsize=12)
axes[0].set_xlabel('Prompt Engineering Skill Level', fontsize=11)
axes[0].set_ylabel('Mean Skill Retention Score', fontsize=11)
axes[0].set_ylim(65, 88)
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Policy', bbox_to_anchor=(1.02,1), loc='upper left', fontsize=9)

# Right: Retention by Tool Diversity
td = df.groupby('Tool_Diversity')['Skill_Retention_Score'].mean()
td_err = df.groupby('Tool_Diversity')['Skill_Retention_Score'].sem()
axes[1].bar(td.index, td.values, yerr=td_err.values, capsize=4,
            color='#673AB7', edgecolor='black', alpha=0.85)
axes[1].set_title('Mean Skill Retention\nby Number of AI Tools Used',
                  fontweight='bold', fontsize=12)
axes[1].set_xlabel('Tool Diversity (# of AI Tools)', fontsize=11)
axes[1].set_ylabel('Mean Skill Retention Score', fontsize=11)
axes[1].set_ylim(68, 84)
axes[1].set_xticks([1,2,3,4,5])

plt.suptitle('RQ3: Smart AI Use and Skill Retention', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('rq3_retention.pdf', bbox_inches='tight')
plt.savefig('rq3_retention.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as rq3_retention.pdf")

## Summary of Key Findings

| Research Question | Key Result |
|---|---|
| **RQ1 – Substitution & GPA** | GenAI displaces traditional study (r = −0.16). Moderate use (3–12h/wk) peaks GPA; heavy use erodes it. |
| **RQ2 – Anxiety & Burnout** | AI dependency predicts anxiety (r = +0.31). Strict bans *increase* anxiety (+0.77 pts) and high-burnout risk (+6%). |
| **RQ3 – Smart Use & Retention** | Prompt engineering skill explains an 11-point retention gap (71 → 82). Tool diversity independently adds ~7 points. |

**Policy Implication:** Institutions should move toward structured, permissive policies that invest in prompt engineering training rather than blanket bans, which demonstrably worsen both mental health and learning retention.